In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

In [2]:
print("Loading dataset...")
df = pd.read_csv('Agriculture_price_dataset.csv')

Loading dataset...


In [3]:
print("Processing date features...")
df['Price Date'] = pd.to_datetime(df['Price Date'])
df['Year'] = df['Price Date'].dt.year
df['Month'] = df['Price Date'].dt.month
df['Day'] = df['Price Date'].dt.day 
df['Quarter'] = df['Price Date'].dt.quarter 
df = df.drop('Price Date', axis=1)
df['Commodity_Variety'] = df['Commodity'].astype(str) + "_" + df['Variety'].astype(str)

Processing date features...


In [4]:
cat_features = ['STATE', 'District Name', 'Market Name', 'Commodity', 'Variety', 'Grade', 'Commodity_Variety']

In [5]:
for col in cat_features:
    df[col] = df[col].astype(str)

In [6]:
x = df.drop(["Modal_Price", "Min_Price", "Max_Price"], axis=1)
y = df["Modal_Price"]

In [7]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.33, random_state=42)

In [8]:
print("Initializing CatBoost Model...")
model = CatBoostRegressor(
    iterations=3000,       
    learning_rate=0.06,   
    depth=10,             
    l2_leaf_reg=5,         
    loss_function='RMSE',
    eval_metric='R2',
    random_seed=42,
    verbose=100            
)

Initializing CatBoost Model...


In [9]:
print("Starting Model Training (It may take a few minutes)...")
model.fit(
    x_train, y_train,
    cat_features=cat_features, 
    eval_set=(x_test, y_test),
    use_best_model=True
)

Starting Model Training (It may take a few minutes)...
0:	learn: 0.0579431	test: 0.0484516	best: 0.0484516 (0)	total: 377ms	remaining: 18m 50s
100:	learn: 0.7177191	test: 0.6127318	best: 0.6127318 (100)	total: 28.4s	remaining: 13m 34s
200:	learn: 0.7636579	test: 0.6428336	best: 0.6428336 (200)	total: 1m	remaining: 14m
300:	learn: 0.7866811	test: 0.6595749	best: 0.6595768 (299)	total: 1m 30s	remaining: 13m 32s
400:	learn: 0.8068299	test: 0.6677870	best: 0.6678197 (399)	total: 2m 4s	remaining: 13m 23s
500:	learn: 0.8164771	test: 0.6729894	best: 0.6729894 (500)	total: 2m 35s	remaining: 12m 55s
600:	learn: 0.8259245	test: 0.6783757	best: 0.6785511 (596)	total: 3m 7s	remaining: 12m 26s
700:	learn: 0.8325716	test: 0.6812312	best: 0.6812312 (700)	total: 3m 38s	remaining: 11m 55s
800:	learn: 0.8382617	test: 0.6830880	best: 0.6830994 (797)	total: 4m 9s	remaining: 11m 24s
900:	learn: 0.8459881	test: 0.6860402	best: 0.6860402 (900)	total: 4m 40s	remaining: 10m 53s
1000:	learn: 0.8537866	test: 0.6

CatBoostRegressor(depth=10, eval_metric='R2', iterations=3000, l2_leaf_reg=5, learning_rate=0.06, loss_function='RMSE', random_seed=42, verbose=100)

In [10]:
print("\nEvaluating Model...")
y_pred = model.predict(x_test)

print("-" * 40)
print("New Honest R2 Score :", r2_score(y_test, y_pred))
print("New MAE (Mean Absolute Error) :", mean_absolute_error(y_test, y_pred))
print("-" * 40)


Evaluating Model...
----------------------------------------
New Honest R2 Score : 0.700692559567293
New MAE (Mean Absolute Error) : 232.66316736217988
----------------------------------------
